# Model Data Preparation

This notebook demonstrates how to take processed experimental data (output of
[`data_processing_example.ipynb`](data_processing_example.ipynb)) and prepare it
for training a surrogate ensemble model with `cleo-optimize-train`.

The training script expects a CSV with at minimum:
| column | description |
|---|---|
| `sequence` | Full-length amino acid sequence |
| `activity` | Z-score normalized activity value |
| `validation` | `1` if held out for validation, `0` for training |

We walk through:
1. Loading the processed & filtered experimental data
2. Choosing which activity metric to train on
3. Z-score normalizing the activity values
4. Splitting into train / validation sets
5. Exporting the final CSV for model training

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

## 1. Load Processed Experimental Data

We start from the filtered CSV produced by the data processing pipeline.
Each row is one construct (aggregated across replicates) with columns for
expression, kinetic rate, and normalized activity.

In [ ]:
data_dir = Path("../example_data/data_processing/output")
raw_df = pd.read_csv(data_dir / "260128_processed_filtered_data_round4.csv")

print(f"Loaded {len(raw_df)} filtered constructs")
print(f"Columns: {list(raw_df.columns)}")
raw_df.head()

## 2. Choose the Activity Metric

The processed data may contain multiple normalized activity columns (e.g.
`g20_norm_rate` for activity relative to the g20 parent, `momi_norm_rate` for
activity relative to MoMI). Choose the metric that best reflects the function
you want to optimize.

Here we use `g20_norm_rate` — the catalytic rate normalized to the g20
positive control.

In [ ]:
activity_col = "g20_norm_rate"

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(raw_df[activity_col], bins=40, edgecolor="black", alpha=0.7)
axes[0].set_xlabel(activity_col)
axes[0].set_ylabel("Count")
axes[0].set_title("Activity Distribution (raw)")

axes[1].hist(raw_df["mscarlett_um_mean"], bins=40, edgecolor="black", alpha=0.7, color="salmon")
axes[1].set_xlabel("Expression (µM)")
axes[1].set_ylabel("Count")
axes[1].set_title("Expression Distribution")

plt.tight_layout()
plt.show()

print(f"Activity stats:")
print(raw_df[activity_col].describe())

## 3. Z-Score Normalize Activity

The surrogate model trains with a Gaussian NLL loss, so z-score normalization
helps the model converge and makes the learned variance estimates more
interpretable. We save the mean and standard deviation so we can map
predictions back to the original scale later.

In [ ]:
activity_mean = raw_df[activity_col].mean()
activity_std = raw_df[activity_col].std()

raw_df["activity"] = (raw_df[activity_col] - activity_mean) / activity_std

print(f"Z-score parameters: mean = {activity_mean:.4f}, std = {activity_std:.4f}")
print(f"Normalized activity range: [{raw_df['activity'].min():.2f}, {raw_df['activity'].max():.2f}]")

plt.figure(figsize=(6, 4))
plt.hist(raw_df["activity"], bins=40, edgecolor="black", alpha=0.7)
plt.xlabel("Activity (z-scored)")
plt.ylabel("Count")
plt.title("Normalized Activity Distribution")
plt.tight_layout()
plt.show()

## 4. Train / Validation Split

We hold out ~15% of constructs as a validation set to tune hyperparameters.
Once you are satisfied with model convergence, retrain on **all** data
(set `use_validation: false` in the config) before proposing the next batch.

In [ ]:
val_fraction = 0.15

np.random.seed(42)
n = len(raw_df)
val_mask = np.zeros(n, dtype=int)
val_indices = np.random.choice(n, size=int(n * val_fraction), replace=False)
val_mask[val_indices] = 1

raw_df["validation"] = val_mask

n_train = (raw_df["validation"] == 0).sum()
n_val = (raw_df["validation"] == 1).sum()
print(f"Train: {n_train}  |  Validation: {n_val}  |  Total: {n}")

### Inspect the split

Verify that the train and validation distributions look similar.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(
    raw_df.loc[raw_df["validation"] == 0, "activity"],
    bins=30, alpha=0.6, label="Train", edgecolor="black",
)
ax.hist(
    raw_df.loc[raw_df["validation"] == 1, "activity"],
    bins=30, alpha=0.6, label="Validation", edgecolor="black",
)
ax.set_xlabel("Activity (z-scored)")
ax.set_ylabel("Count")
ax.set_title("Train / Validation Split")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Export Training CSV

Select only the columns required by the training pipeline and save.
Point your training config's `data_path` to this CSV.

In [ ]:
output_dir = Path("../example_data/data_processing/output")

export_df = raw_df[["sequence", "name", "activity", "validation"]].copy()

export_path = output_dir / "model_training_dataset.csv"
export_df.to_csv(export_path, index=False)

print(f"Saved {len(export_df)} rows to {export_path}")
export_df.head()

## Next Steps

With the training CSV ready you can now:

1. **Train with validation** to tune hyperparameters:
   ```bash
   cleo-optimize-train --config-name momi use_validation=true data_path=<path_to_csv>
   ```

2. **Evaluate** the model using the
   [`model_training_and_evaluation.ipynb`](model_training_and_evaluation.ipynb)
   notebook.

3. **Retrain on all data** (set `use_validation=false`) once hyperparameters
   are tuned, then proceed to batch optimization.

See the [README](../README.md#-training-sequence-to-function-models) for full
details on the training workflow.